# GenAI Assignment 1 - training on Kaggle (2 x T4)
Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**. Add-ons -> Secrets: add `WANDB_API_KEY`.
Then **Save Version -> Save & Run All (Commit)** so it keeps running when the browser is closed.
Tasks 1-3 run on GPU 0 and Task 4 on GPU 1 in parallel. Download `/kaggle/working/results.zip` from the Output tab when it finishes.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B key loaded")
except Exception as e:
    print("No W&B secret - logging offline:", e)

In [ ]:
REPO = "https://github.com/ibrahimmlk/Generative-AI-Assignment-1.git"
!git clone -q $REPO /kaggle/working/genai-a1
%cd /kaggle/working/genai-a1
!pip install -q optuna wandb onnx onnxscript onnxruntime gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, time
W = "/kaggle/working"
env0 = dict(os.environ, CUDA_VISIBLE_DEVICES="0", NUM_WORKERS="3", PYTHONUNBUFFERED="1")
env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="1", NUM_WORKERS="1", PYTHONUNBUFFERED="1")
p1 = subprocess.Popen(f"python scripts/run_restoration.py --data {W}/data --out {W}/outputs --budget full > {W}/restoration.log 2>&1", shell=True, env=env0)
p2 = subprocess.Popen(f"python scripts/run_sketch.py --fs2k {W}/data/FS2K --data {W}/data --out {W}/outputs_sketch --budget full > {W}/sketch.log 2>&1", shell=True, env=env1)
while p1.poll() is None or p2.poll() is None:
    time.sleep(300)
    for name in ["restoration", "sketch"]:
        print(f"===== {name} =====")
        os.system(f"tail -n 4 {W}/{name}.log")
print("exit codes:", p1.returncode, p2.returncode)

In [ ]:
!tail -n 60 /kaggle/working/restoration.log
!tail -n 30 /kaggle/working/sketch.log

In [ ]:
%cd /kaggle/working
!zip -q -r results.zip outputs outputs_sketch restoration.log sketch.log -x "*/wandb/*"
!ls -la results.zip